## 1. Load and Inspect Silver Air Quality Data

Load the cleaned PM2.5 measurements from the Silver layer and inspect the schema and sample records before daily aggregation.

In [ ]:
from pyspark.sql import functions as F

df_air = spark.table("dbo.silver_air_quality")

In [ ]:
df_air.printSchema()
df_air.show(5)

## 2. Aggregate Daily PM2.5 Measurements

Aggregate hourly PM2.5 measurements by date and sensor. Calculate the daily average concentration and the number of valid measurements per day.

In [ ]:
df_air_daily = df_air.groupBy(
    "measurement_date",
    "sensor_id"
).agg(
    F.avg("pm25_value").alias("avg_pm25"),
    F.count("pm25_value").alias("measurement_count")
)

In [ ]:
df_air_daily.orderBy("measurement_date").show(5)

In [ ]:
df_air_daily.groupBy(
    "measurement_count"
).count().orderBy("measurement_count").show()

## 3. Validate Daily Air Quality Fact

Perform data quality checks to verify row counts, uniqueness of the measurement_date and sensor_id combination, and absence of NULL values in critical columns.

In [ ]:
print("Total rows:", df_air_daily.count())

print(
    "Unique keys:",
    df_air_daily.select(
        "measurement_date",
        "sensor_id"
    ).distinct().count()
)

In [ ]:
df_air_daily.selectExpr(
    "SUM(CASE WHEN measurement_date IS NULL THEN 1 ELSE 0 END) AS null_dates",
    "SUM(CASE WHEN sensor_id IS NULL THEN 1 ELSE 0 END) AS null_sensors",
    "SUM(CASE WHEN avg_pm25 IS NULL THEN 1 ELSE 0 END) AS null_pm25"
).show()

## 4. Save and Verify Gold Air Quality Fact

Save the validated daily PM2.5 aggregates as a Delta table in the Gold layer and verify the persisted records.

In [ ]:
df_air_daily.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dbo.gold_fact_air_quality_daily")

In [ ]:
df_saved_air = spark.table("dbo.gold_fact_air_quality_daily")

print("Saved rows:", df_saved_air.count())

df_saved_air.orderBy("measurement_date").show(5)